# 02 — Reddit Preprocessing

**Stage:** flatten curated Reddit thread JSON into a cleaned, language-filtered,
deduplicated comment table that downstream notebooks can consume.

**Inputs:** `data/raw/reddit/reddit_*.json` (gitignored — raw collector pulls).

**Outputs (`data/processed/01_preprocessed/`):**
- `reddit_comments.csv` (+ `.parquet`) — one row per comment with raw text, two cleaned
  versions, lemma tokens, hashed author + hashed parent author, plus the Reddit-specific
  metadata (depth, score, subreddit, is_submitter, distinguished, controversiality,
  is_removed) that downstream notebooks need.
- `reddit_submissions.csv` — one row per submission (subreddit, title, score, ...).
- `reddit_preprocessing_summary.json` — provenance + counts + per-step attrition.

## Course-pillar mapping (for the rubric)

This notebook implements **CLO 1** (apply data science to social-media data) at the
data-preparation stage. The text pipeline mirrors the **W2/W3 canonical `processText()`**
(lower-case → TweetTokenize → lemmatise → drop NLTK English stopwords + digits + 1-char
tokens) and is *intentionally identical* to notebook 01, so Reddit and YouTube comments
end up directly comparable downstream (nb 03 vocabulary, nb 04 network, nb 05 sentiment +
topics, nb 06 cascade, nb 07 cross-platform tests). The only deliberate adaptations are:

- **Reddit-specific flatten** that walks `threads[*].comments[*]` and reads `body`
  instead of `text`, parses `created_utc` from unix epoch, and resolves the reply edge
  via `parent_author` (rather than `parent_id`-pointing-to-comment) so the reply graph
  in notebook 04 has a directed `replier → parent_author` edge for every reply.
- **Reddit metadata preserved** in the final dataframe — `depth`, `score`,
  `controversiality`, `is_submitter`, `distinguished`, `is_removed`, `comment_name`
  (`t1_*` full name), `parent_id` (`t1_*` or `t3_*`).

No methods from outside the course are introduced.

## Why Reddit is the project's *primary* substrate

Reddit reply trees are deep — `depth` can reach 6–8 in our threads. That makes the
Reddit reply graph the right substrate for influence analysis (PageRank on directed
reply edges in nb 04), community detection (Louvain on the undirected weighted
projection), and diffusion / cascade analysis (time-to-first-reply, half-life,
branching in nb 06). YouTube's reply structure is single-level by platform design,
which is why the report treats YouTube as *comparison*, not as the primary network.

## Pipeline overview

1. **Setup** — paths, imports, NLTK resources (identical to nb 01).
2. **Load raw JSON** — flatten threads + submissions + comments; project-window filter early.
3. **Initial exploration BEFORE cleaning** — top-30 raw tokens + sample comments.
4. **Language detection + filtering** — same `detect_language_improved` as nb 01.
5. **Reddit moderation/bot-template filter** — remove AutoModerator / VisualMod and
   strong subreddit governance templates before they affect token frequencies.
6. **Text cleaning — two pipelines** — identical `clean_text` (aggressive) and
   `clean_text_for_vader` (light) as in nb 01.
7. **Duplicate analysis** — exact + near-duplicate, no per-author cap.
8. **Tokenisation + base stopwords** — NLTK English + punctuation; print top-30 to inspect.
9. **Domain-specific stopwords** — list informed by the top-30 print; identical core list
   to nb 01 with Reddit-UI noise instead of YouTube-UI noise.
10. **Worked example** — one comment traced through every stage.
11. **Visualisations** — top tokens, length distribution, subreddit + reply-depth, word
    cloud, comments over time.
12. **Write outputs + summary JSON.**
13. **Limitations.**


## Setup


In [ ]:
"""Setup — paths, imports, NLTK resources.

Identical to notebook 01 so the two preprocessing notebooks share the same
environment. Heavy NLTK assets are downloaded on first run.
"""

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA = PROJECT_ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
PREPROCESSED = PROCESSED / "01_preprocessed"
PREPROCESSED.mkdir(parents=True, exist_ok=True)
PLOTS = PROJECT_ROOT / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

# --- Standard imports --------------------------------------------------------
import hashlib
import hmac
import json
import re
import string
import unicodedata
from collections import Counter
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- NLP imports -------------------------------------------------------------
import nltk
from nltk.corpus import stopwords as _nltk_stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import TweetTokenizer

from langdetect import DetectorFactory, LangDetectException, detect

try:
    import emoji
except ImportError:
    emoji = None  # only used by the worked-example trace; not critical

from wordcloud import WordCloud

# --- Project config ----------------------------------------------------------
from config import (
    PROJECT_END_DATE,
    PROJECT_START_DATE,
    RAW_REDDIT_DIR,
    get_anon_salt,
)

# --- NLTK resources ----------------------------------------------------------
for resource, path in [
    ("stopwords", "corpora/stopwords"),
    ("wordnet", "corpora/wordnet"),
    ("omw-1.4", "corpora/omw-1.4"),
]:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(resource, quiet=True)

# Make langdetect deterministic.
DetectorFactory.seed = 0

# Notebook display settings.
%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

print(f"Project root : {PROJECT_ROOT}")
print(f"Project window: {PROJECT_START_DATE}  →  {PROJECT_END_DATE}")


## 1 · Load raw Reddit JSON

We walk every thread in every raw JSON file under `data/raw/reddit/`. The raw shape
is `{collected_at_utc, source, parameters, counts, threads:[{submission:{...},
comments:[...], ...}]}`. We dedupe by `submission_id` and by `comment_id` so multiple
collection runs can be layered safely.

**Reddit-specific decisions:**

- Comment text is in `body`, not `text`.
- Timestamps are unix epoch seconds (`created_utc`), converted with `unit='s'`.
- There is no `is_reply` boolean — we derive it from `depth > 0`.
- Engagement is `score` (upvotes − downvotes); there are no separate like/reply counts.
- For the reply graph (nb 04) we keep BOTH `parent_id` (the Reddit `t1_*` / `t3_*`
  full name of the parent) and `parent_author_hash` (the hashed author of that parent).
  The collector already resolves `parent_author` to a username even when the parent
  is the submission itself, so we hash it the same way as the comment author.
- We preserve every Reddit-specific metadata field (`depth`, `score`, `controversiality`,
  `is_submitter`, `distinguished`, `is_removed`, `comment_name`) because downstream
  notebooks rely on them for diffusion analysis (nb 06) and influencer typology (nb 04).

The project-window filter is applied **here** — there's no point doing NLP on
comments outside the window.


In [ ]:
def hash_author(author, salt):
    """HMAC-SHA256(salt, author) → 16-hex string, or None for missing/deleted authors.

    HMAC with a secret salt is irreversible without the salt (resistant to rainbow
    tables on common Reddit usernames). The salt lives outside the repository.
    """
    if not author or author.strip().lower() in {"[deleted]", "deleted", "anonymous"}:
        return None
    return hmac.new(
        salt.encode("utf-8"),
        author.encode("utf-8"),
        hashlib.sha256,
    ).hexdigest()[:16]


def _row_for_submission(sub, thread_meta, salt):
    return {
        "submission_id": sub.get("submission_id"),
        "subreddit": sub.get("subreddit"),
        "title": sub.get("title", ""),
        "selftext": sub.get("selftext", ""),
        "author_hash": hash_author(sub.get("author"), salt),
        "created_utc": sub.get("created_utc"),
        "score": sub.get("score"),
        "num_comments": sub.get("num_comments"),
        "permalink": sub.get("permalink"),
        "url": sub.get("url"),
        "collected_at_utc": thread_meta.get("collected_at_utc"),
        "collected_comment_count": thread_meta.get("collected_comment_count"),
    }


def _row_for_comment(c, sub, salt):
    body = c.get("body") or ""
    author_lower = (c.get("author") or "").strip().lower()
    return {
        "platform": "reddit",
        "submission_id": c.get("submission_id") or sub.get("submission_id"),
        "subreddit": sub.get("subreddit") if sub else None,
        "comment_id": c.get("comment_id"),
        "comment_name": c.get("name"),            # Reddit full name, t1_*
        "parent_id": c.get("parent_id"),          # t1_* (parent comment) or t3_* (submission)
        "author_hash": hash_author(c.get("author"), salt),
        "parent_author_hash": hash_author(c.get("parent_author"), salt),  # for nb 04 reply graph
        "raw_text": body,                                                  # verbatim, for inspection
        "depth": c.get("depth"),
        "is_submitter": c.get("is_submitter"),
        "distinguished": c.get("distinguished"),
        "score": c.get("score"),
        "controversiality": c.get("controversiality"),
        "is_removed": c.get("removed"),
        # Internal preprocessing flag only; raw author names are not written out.
        "is_moderation_bot": author_lower in {"automoderator", "visualmod"},
        "created_utc": c.get("created_utc"),
    }


def flatten_reddit_raw(raw_paths, salt, start_date, end_date):
    """Flatten raw Reddit thread JSON → (rd_subs_df, rd_raw_df, collection_meta).

    De-duplicates by submission_id / comment_id so layering multiple collection runs
    (e.g. an early baseline plus a fresh pull) is safe.

    Comments outside ``[start_date, end_date]`` (inclusive) are dropped immediately,
    before any NLP — saves work and keeps attrition counts clean.
    """
    sub_rows, comment_rows = [], []
    seen_subs, seen_comments = set(), set()
    collection_meta = {}

    for path in raw_paths:
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError) as exc:
            print(f"[warn] cannot read {path}: {exc}")
            continue
        # Capture the latest run's provenance.
        if "collected_at_utc" in data:
            collection_meta = {
                "collected_at_utc": data.get("collected_at_utc"),
                "source": data.get("source"),
                "parameters": data.get("parameters", {}),
                "counts": data.get("counts", {}),
            }
        run_meta = {"collected_at_utc": data.get("collected_at_utc")}
        for thread in data.get("threads", []):
            sub = thread.get("submission") or {}
            sid = sub.get("submission_id")
            if not sid or sid in seen_subs:
                continue
            seen_subs.add(sid)
            comments_here = thread.get("comments") or []
            thread_meta = {**run_meta, "collected_comment_count": len(comments_here)}
            sub_rows.append(_row_for_submission(sub, thread_meta, salt))
            for c in comments_here:
                cid = c.get("comment_id")
                if not cid or cid in seen_comments:
                    continue
                seen_comments.add(cid)
                comment_rows.append(_row_for_comment(c, sub, salt))

    rd_subs_df = pd.DataFrame(sub_rows)
    rd_raw_df = pd.DataFrame(comment_rows)

    # Parse Reddit timestamps (unix epoch seconds).
    if not rd_raw_df.empty:
        rd_raw_df["created_at"] = pd.to_datetime(
            rd_raw_df["created_utc"], unit="s", utc=True, errors="coerce"
        )
        rd_raw_df["date"] = rd_raw_df["created_at"].dt.date.astype("string")

        # Project-window filter, applied EARLY.
        start_ts = pd.Timestamp(start_date, tz="UTC")
        end_ts = (pd.Timestamp(end_date, tz="UTC")
                  + pd.Timedelta(days=1) - pd.Timedelta(microseconds=1))
        before = len(rd_raw_df)
        rd_raw_df = rd_raw_df[
            (rd_raw_df["created_at"] >= start_ts)
            & (rd_raw_df["created_at"] <= end_ts)
        ].copy()
        print(f"  Project-window filter: kept {len(rd_raw_df):,} / {before:,} comments")

    # Same parse + window restriction for submissions (so the two stay in sync).
    if not rd_subs_df.empty:
        rd_subs_df["created_at"] = pd.to_datetime(
            rd_subs_df["created_utc"], unit="s", utc=True, errors="coerce"
        )
        rd_subs_df["date"] = rd_subs_df["created_at"].dt.date.astype("string")
        if not rd_raw_df.empty:
            keep = set(rd_raw_df["submission_id"].dropna())
            rd_subs_df = rd_subs_df[rd_subs_df["submission_id"].isin(keep)].copy()

    return rd_subs_df, rd_raw_df, collection_meta


In [ ]:
raw_paths = sorted(RAW_REDDIT_DIR.glob("reddit_*.json"))
# Skip per-run summary files that the collector sometimes writes alongside.
raw_paths = [p for p in raw_paths if not p.name.endswith("_summary.json")]
if not raw_paths:
    raise FileNotFoundError(
        "No raw Reddit JSON found under data/raw/reddit/. "
        "Run `python3 src/reddit_collect.py` first."
    )

print(f"Loading {len(raw_paths)} raw JSON file(s):")
for p in raw_paths:
    print(f"  - {p.name}")

salt = get_anon_salt()
rd_subs_df, rd_raw_df, collection_meta = flatten_reddit_raw(
    raw_paths, salt, PROJECT_START_DATE, PROJECT_END_DATE
)

# Per-step attrition tally; we append to it after every filter.
attrition = []
attrition.append({"step": "raw_loaded", "comments": int(len(rd_raw_df))})

print()
print(f"After raw load + window filter:")
print(f"  submissions     : {len(rd_subs_df):,}")
print(f"  comments        : {len(rd_raw_df):,}")
print(f"  unique authors  : {rd_raw_df['author_hash'].nunique():,}")
print(f"  unique subreddits: {rd_raw_df['subreddit'].nunique():,}")
print(f"  unique submissions in comment table: {rd_raw_df['submission_id'].nunique():,}")
date_min, date_max = rd_raw_df["created_at"].min(), rd_raw_df["created_at"].max()
print(f"  date range      : {date_min}  →  {date_max}")
print(f"  reply-depth max : {int(rd_raw_df['depth'].max())}")
print(f"  is_removed=True : {int(rd_raw_df['is_removed'].fillna(False).sum())}")


## 2 · Initial exploration BEFORE cleaning

Three random raw comments and the top-30 raw token frequencies. As with notebook 01,
the unfiltered top is dominated by stopwords and punctuation — by design — and that
motivates the cleaning rules in §4.


In [ ]:
# Three random raw comments.
print("Random raw comments (no cleaning):\n")
for i, row in rd_raw_df.sample(3, random_state=7).iterrows():
    txt = row["raw_text"] or ""
    print(f'  subreddit={row["subreddit"]}  depth={row["depth"]}  len={len(txt)}')
    print(f'    {txt[:240]!r}')
    print()

# Raw token frequencies via TweetTokenizer on the unprocessed body.
_tweet_tok = TweetTokenizer(reduce_len=True, strip_handles=False)
raw_counter = Counter()
for txt in rd_raw_df["raw_text"].fillna(""):
    raw_counter.update(t.lower() for t in _tweet_tok.tokenize(txt))

print(f"Unique raw tokens: {len(raw_counter):,}\n")
print("Top-30 raw tokens (dominated by stopwords + Markdown punctuation — by design):\n")
for i, (tok, n) in enumerate(raw_counter.most_common(30), 1):
    pct = n / len(rd_raw_df) * 100
    print(f"  {i:3d}. {tok:25s} {n:>7,}   (in ~{pct:.1f}% of comments)")


## 3 · Language detection and filtering

Identical detector to notebook 01: `detect_language_improved` uses three rules
tuned for short social-media text — (1) short Latin-script defaults to English,
(2) significant non-Latin script triggers `langdetect`, (3) longer Latin-script
requires three consistent non-English detections. Returns `'en'`, an ISO code,
or `'unknown'` on detector errors.


In [ ]:
def detect_language_improved(text):
    """Three-rule language detector tuned for short social-media text.

    Returns
    -------
    str
        ISO 639-1 language code, or ``"unknown"`` when the detector raised.
    """
    if not isinstance(text, str):
        return "unknown"
    text_stripped = text.strip()
    if not text_stripped:
        return "unknown"

    non_latin_chars = sum(
        1 for ch in text_stripped
        if ord(ch) > 0x024F
        and not (0x2000 <= ord(ch) <= 0x27FF)
        and not (0x1F600 <= ord(ch) <= 0x1FAFF)
    )
    non_latin_ratio = non_latin_chars / max(len(text_stripped), 1)

    if non_latin_ratio > 0.3 and len(text_stripped) > 5:
        try:
            return detect(text_stripped)
        except LangDetectException:
            return "unknown"

    if len(text_stripped) < 50:
        return "en"

    try:
        results = [detect(text_stripped) for _ in range(3)]
        if all(lang != "en" for lang in results) and len(set(results)) == 1:
            return results[0]
        else:
            return "en"
    except LangDetectException:
        return "unknown"


def inspect_language(df, lang_code, n=5, max_chars=200, seed=0):
    """Print up to ``n`` random comments classified as ``lang_code``."""
    sub = df[df["language"] == lang_code]
    if sub.empty:
        print(f"  (no comments classified as {lang_code})")
        return
    sample_n = min(n, len(sub))
    samples = sub.sample(sample_n, random_state=seed)
    print(f"Language = {lang_code!r}  ({len(sub)} comments total)\n")
    for _, row in samples.iterrows():
        print(f"  - {(row['raw_text'] or '')[:max_chars]!r}")


In [ ]:
# Apply the detector. Reddit comments are longer on average than YouTube comments,
# so langdetect's per-row cost is similar.
rd_raw_df["language"] = rd_raw_df["raw_text"].apply(detect_language_improved)

lang_counts = rd_raw_df["language"].value_counts()
print("Top-12 detected languages:\n")
for lang, n in lang_counts.head(12).items():
    pct = n / len(rd_raw_df) * 100
    print(f"  {lang:8s} {n:>7,}   ({pct:.2f}%)")


In [ ]:
# Inspect a small sample from the largest non-English categories.
non_en = rd_raw_df[rd_raw_df["language"] != "en"]
print(f"Non-English comments: {len(non_en):,}\n")
print(f"Distinct non-English language codes: {non_en['language'].nunique()}\n")

for code in non_en["language"].value_counts().head(4).index:
    print("=" * 60)
    inspect_language(rd_raw_df, code, n=3)
    print()


In [ ]:
# Filter to English-only.
before = len(rd_raw_df)
rd_en_df = rd_raw_df[rd_raw_df["language"] == "en"].copy()
attrition.append({"step": "language_en_only", "comments": int(len(rd_en_df))})

print(f"Dropped {before - len(rd_en_df):,} non-English comments "
      f"({(before - len(rd_en_df)) / before * 100:.2f}%)")
print(f"Remaining English comments: {len(rd_en_df):,}")


## 3b · Reddit moderation / bot-template filter

Some Reddit comments are not user discourse: AutoModerator removal notices, WSB
profile/report tables, and subreddit rule reminders. These survive the normal
`[removed]` / `[deleted]` filter because they contain English text, but they would
pollute token frequencies, topics, sentiment, and network analysis.

This filter is deliberately conservative. It removes known moderation-bot authors
(`AutoModerator`, `VisualMod`) and strong template phrases, but it does **not**
remove ordinary comments merely because they contain the word “removed”.


In [ ]:
# Strong Reddit governance / bot-template patterns.
_REDDIT_BOT_TEMPLATE_RE = re.compile(
    r"your submission (?:has been |was )?automatically removed|"
    r"your comment (?:has been |was )?removed because|"
    r"this action was performed automatically|"
    r"\*\*user report\*\*|"
    r"total submissions.*first seen in wsb|"
    r"previous best dd|"
    r"join wsb discord",
    flags=re.IGNORECASE | re.DOTALL,
)

bot_noise_mask = (
    rd_en_df["is_moderation_bot"].fillna(False)
    | rd_en_df["raw_text"].fillna("").str.contains(_REDDIT_BOT_TEMPLATE_RE)
)

reddit_noise_df = rd_en_df[bot_noise_mask].copy()
before = len(rd_en_df)
rd_en_df = rd_en_df[~bot_noise_mask].copy()
attrition.append({"step": "reddit_moderation_bot_noise_drop", "comments": int(len(rd_en_df))})

reddit_bot_noise_summary = {
    "removed_comments": int(before - len(rd_en_df)),
    "known_bot_author_comments": int(reddit_noise_df["is_moderation_bot"].fillna(False).sum()),
    "template_pattern_comments": int(
        reddit_noise_df["raw_text"].fillna("").str.contains(_REDDIT_BOT_TEMPLATE_RE).sum()
    ),
    "known_bot_authors": ["AutoModerator", "VisualMod"],
    "template_patterns": [
        "your submission has/was automatically removed",
        "your comment was removed because",
        "this action was performed automatically",
        "**user report** / WSB account tables",
        "join wsb discord / previous best dd",
    ],
}

print(f"Removed {before - len(rd_en_df):,} Reddit moderation/bot-template comments")
print(f"Remaining English user comments: {len(rd_en_df):,}\n")

if reddit_noise_df.empty:
    print("No moderation/bot-template comments matched.")
else:
    print("Examples removed by the moderation/bot-template filter:")
    for _, row in reddit_noise_df[["subreddit", "raw_text"]].head(8).iterrows():
        print(f"\n--- r/{row['subreddit']} ---")
        print((row["raw_text"] or "")[:360].replace("\n", " "))

# Guardrail: the remaining English corpus should have no strong bot/template hits.
remaining_template_hits = int(
    rd_en_df["raw_text"].fillna("").str.contains(_REDDIT_BOT_TEMPLATE_RE).sum()
)
print(f"\nRemaining strong template-pattern hits: {remaining_template_hits}")


## 4 · Text cleaning — two pipelines

**Identical to notebook 01.** Two cleaned versions per comment:

| Column | Used by | Style |
|---|---|---|
| `clean_text` | topic models, log-odds vocabulary, frequency, lexicons | Aggressive — strips URLs, mentions, emojis, timestamps; lowercases. |
| `vader_text` | VADER + transformer sentiment | Light — preserves capitalisation and punctuation. |

Both functions:

- Normalise dotted single-letter acronyms (`U.S.A.` → `USA`, `U.S.` → `USA`, `US` → `USA`,
  `U.K.` → `UK`, `E.U.` → `EU`, `U.N.` → `UN`, `N.A.T.O.` → `NATO`). Mapping `US`/`U.S.`
  to the canonical `USA` lets us safely add the pronoun `us` to the stopword list in §7
  without losing country mentions.
- Collapse 2+ identical punctuation marks to 1 (`!!!` → `!`, `,,,` → `,`).
- Strip zero-width characters (`\u200b`/`\u200c`/`\u200d`/`\ufeff`) that sometimes leak
  in from copy-paste.


In [ ]:
# Pre-compile patterns (cheaper than re.sub with a string pattern on every row).
_URL_RE = re.compile(r"http\S+|www\.\S+")
_HTML_ENTITY_RE = re.compile(r"&\w+;")
_HTML_TAG_RE = re.compile(r"<[^>]+>")
_MENTION_RE = re.compile(r"@\w+")
_TIMESTAMP_RE = re.compile(r"\b\d{1,2}:\d{2}(?::\d{2})?\b")
_REPEAT_PUNCT_RE = re.compile(r"([!?.,])\1+")
_WHITESPACE_RE = re.compile(r"\s+")
_ZERO_WIDTH_RE = re.compile(r"[\u200b\u200c\u200d\ufeff]")
_EMOJI_RE = re.compile(
    "["
    "\U0001F600-\U0001F64F"   # emoticons
    "\U0001F300-\U0001F5FF"   # symbols & pictographs
    "\U0001F680-\U0001F6FF"   # transport & map symbols
    "\U0001F1E0-\U0001F1FF"   # flags
    "\U00002702-\U000027B0"   # dingbats
    "\U000024C2-\U0001F251"
    "\U0001f926-\U0001f937"
    "\U0001F900-\U0001F9FF"   # supplemental symbols
    "\U0001FA70-\U0001FAFF"   # extended-A pictographs
    "]+",
    flags=re.UNICODE,
)

_SMART_QUOTES = {
    "\u2018": "'", "\u2019": "'",
    "\u201c": '"', "\u201d": '"',
    "\u2014": " ", "\u2013": " ",
}

# Acronym rules — IDENTICAL to notebook 01.
# US / U.S. / U S are all collapsed to USA so the pronoun "us" can safely
# enter the stopword list in §7 without losing country mentions.
_ACRONYM_RULES = [
    # USA forms
    (re.compile(r"\bU\.\s?S\.\s?A\.?\b", re.IGNORECASE), "USA"),
    (re.compile(r"\bU\s+S\s+A\b", re.IGNORECASE), "USA"),
    (re.compile(r"\bUSA\b"), "USA"),
    # US country forms (all map to USA — canonical)
    (re.compile(r"\bU\.\s?S\.?\b", re.IGNORECASE), "USA"),
    (re.compile(r"\bU\s+S\b", re.IGNORECASE), "USA"),
    (re.compile(r"\bUS\b"), "USA"),
    # Other acronyms
    (re.compile(r"\bU\.\s?K\.?\b", re.IGNORECASE), "UK"),
    (re.compile(r"\bU\s+K\b", re.IGNORECASE), "UK"),
    (re.compile(r"\bUK\b"), "UK"),
    (re.compile(r"\bE\.\s?U\.?\b", re.IGNORECASE), "EU"),
    (re.compile(r"\bE\s+U\b", re.IGNORECASE), "EU"),
    (re.compile(r"\bEU\b"), "EU"),
    (re.compile(r"\bU\.\s?N\.?\b", re.IGNORECASE), "UN"),
    (re.compile(r"\bU\s+N\b", re.IGNORECASE), "UN"),
    (re.compile(r"\bUN\b"), "UN"),
    (re.compile(r"\bN\.\s?A\.\s?T\.\s?O\.?\b", re.IGNORECASE), "NATO"),
    (re.compile(r"\bN\s+A\s+T\s+O\b", re.IGNORECASE), "NATO"),
    (re.compile(r"\bNATO\b"), "NATO"),
]


def normalise_acronyms(text):
    """Collapse common dotted acronyms (U.S., U.K., E.U., …) to canonical bare uppercase."""
    for pat, repl in _ACRONYM_RULES:
        text = pat.sub(repl, text)
    return text


def normalise_smart_punct(text):
    """Replace smart quotes and en/em dashes with ASCII equivalents."""
    for src, tgt in _SMART_QUOTES.items():
        text = text.replace(src, tgt)
    return text


def clean_text(text):
    """Aggressive cleaning for topic / log-odds / frequency analysis.

    Steps:
      1. Normalise dotted acronyms.
      2. Strip URLs, HTML entities, HTML tags, @mentions, timestamps.
      3. Remove emojis.
      4. Normalise smart quotes and dashes.
      5. Collapse 2+ identical punctuation marks to 1.
      6. Collapse whitespace.
      7. Strip zero-width characters.
      8. Lowercase.
    """
    if not isinstance(text, str):
        return ""
    text = normalise_acronyms(text)
    text = _URL_RE.sub(" ", text)
    text = _HTML_ENTITY_RE.sub(" ", text)
    text = _HTML_TAG_RE.sub(" ", text)
    text = _MENTION_RE.sub(" ", text)
    text = _TIMESTAMP_RE.sub(" ", text)
    text = _EMOJI_RE.sub(" ", text)
    text = normalise_smart_punct(text)
    text = _REPEAT_PUNCT_RE.sub(r"\1", text)
    text = _WHITESPACE_RE.sub(" ", text).strip()
    text = _ZERO_WIDTH_RE.sub("", text)
    return text.lower()


def clean_text_for_vader(text):
    """Light cleaning for VADER sentiment.

    Preserves capitalisation and punctuation. Removes only non-content elements
    (URLs, mentions, timestamps, HTML), normalises smart punctuation, collapses
    runs of 2+ identical punctuation marks to 1, strips zero-width characters.
    """
    if not isinstance(text, str):
        return ""
    text = normalise_acronyms(text)
    text = _URL_RE.sub(" ", text)
    text = _HTML_ENTITY_RE.sub(" ", text)
    text = _HTML_TAG_RE.sub(" ", text)
    text = _MENTION_RE.sub(" ", text)
    text = _TIMESTAMP_RE.sub(" ", text)
    text = normalise_smart_punct(text)
    text = _REPEAT_PUNCT_RE.sub(r"\1", text)
    text = _WHITESPACE_RE.sub(" ", text).strip()
    text = _ZERO_WIDTH_RE.sub("", text)
    return text  # capitalisation preserved


In [ ]:
# Apply both cleaners.
rd_en_df["clean_text"] = rd_en_df["raw_text"].apply(clean_text)
rd_en_df["vader_text"] = rd_en_df["raw_text"].apply(clean_text_for_vader)

# Drop rows that collapsed to empty after aggressive cleaning. Reddit's
# "[deleted]" and "[removed]" bodies fall into this bucket once aggressive
# cleaning normalises them away.
before = len(rd_en_df)
rd_clean_df = rd_en_df[rd_en_df["clean_text"].str.len() > 0].copy()
attrition.append({"step": "non_empty_after_clean", "comments": int(len(rd_clean_df))})

print(f"Dropped {before - len(rd_clean_df):,} comments that became empty after aggressive cleaning")
print(f"Remaining: {len(rd_clean_df):,}\n")

# Side-by-side preview of raw / clean / vader text.
print("Sample raw / clean_text / vader_text:")
preview = (rd_clean_df.sample(5, random_state=11)
           [["raw_text", "clean_text", "vader_text"]])
for i, row in preview.iterrows():
    print(f"\n--- comment {i} ---")
    print(f"  RAW   : {(row['raw_text'] or '')[:200]!r}")
    print(f"  CLEAN : {(row['clean_text'] or '')[:200]!r}")
    print(f"  VADER : {(row['vader_text'] or '')[:200]!r}")


## 5 · Duplicate analysis 

Same strategy as notebook 01: exact-duplicate drop on `clean_text`, then
near-duplicate drop on a digit/punctuation-stripped key. **No per-author cap** —
the reply network in nb 04 sees every user's edges.

Reddit's karma incentive against copy-paste means we expect fewer duplicates than
on YouTube. We still run both stages so the comparison is apples-to-apples.


In [ ]:
print("=== Stage 1: Exact-duplicate diagnosis ===\n")
n_exact = rd_clean_df["clean_text"].duplicated(keep=False).sum()
print(f"Comments that are exact duplicates of another: {n_exact:,} "
      f"({n_exact / len(rd_clean_df) * 100:.2f}%)\n")

print("Top-10 most-repeated cleaned texts:")
top_rep = rd_clean_df["clean_text"].value_counts().head(10)
for txt, n in top_rep.items():
    print(f"  [{n}x] {txt[:140]}")


In [ ]:
# Drop exact duplicates (keep the first occurrence by file order).
duplicate_mask = rd_clean_df.duplicated(subset=["clean_text"], keep="first")
duplicate_comments_df = rd_clean_df[duplicate_mask].copy()
before = len(rd_clean_df)
rd_clean_df = rd_clean_df.drop_duplicates(subset=["clean_text"], keep="first").copy()
attrition.append({"step": "exact_duplicate_drop", "comments": int(len(rd_clean_df))})

print(f"Removed {before - len(rd_clean_df):,} exact duplicates")
print(f"Remaining: {len(rd_clean_df):,}")


In [ ]:
def normalise_for_dedup(text):
    """Aggressive normalisation for near-duplicate detection only.

    Removes digits, all punctuation, and collapses whitespace — so two templated
    comments differing only in a date or percentage hash the same.
    """
    text = text.lower()
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


rd_clean_df["_dedup_key"] = rd_clean_df["clean_text"].apply(normalise_for_dedup)

print("=== Stage 2: Near-duplicate diagnosis (after stripping digits / punctuation) ===\n")
n_near = rd_clean_df["_dedup_key"].duplicated(keep=False).sum()
print(f"Near-duplicate comments: {n_near:,}\n")

print("Examples of near-duplicate groups (size > 2):")
groups = (rd_clean_df.groupby("_dedup_key")
          .filter(lambda g: len(g) > 2)
          .groupby("_dedup_key"))
shown = 0
for key, grp in groups:
    if shown >= 3 or not key.strip():
        continue
    print(f"\n  Group of {len(grp)} near-duplicates:")
    for txt in grp["clean_text"].head(3):
        print(f"    - {txt[:140]}")
    shown += 1


In [ ]:
# Drop near-duplicates.
before = len(rd_clean_df)
rd_clean_df = rd_clean_df.drop_duplicates(subset=["_dedup_key"], keep="first").copy()
rd_clean_df = rd_clean_df.drop(columns=["_dedup_key"])
attrition.append({"step": "near_duplicate_drop", "comments": int(len(rd_clean_df))})

print(f"Removed {before - len(rd_clean_df):,} near-duplicates")
print(f"Remaining: {len(rd_clean_df):,}\n")

print("Per-author concentration after both dedup stages:")
print(rd_clean_df["author_hash"].value_counts().head(10))


## 6 · Tokenisation + base stopwords (course pipeline)

Same W2/W3 pipeline as notebook 01: lowercase → `TweetTokenizer` → drop NLTK
English stopwords + punctuation + 1-char tokens + pure-digit tokens → WordNet
noun-default lemmatise. We **keep** 2-character tokens (`eu`, `un`, `uk`, `oil`)
deliberately.


In [ ]:
_tokenizer = TweetTokenizer(reduce_len=True, strip_handles=False)
_lemmatizer = WordNetLemmatizer()

_PUNCT = list(string.punctuation)
_STOPWORDS_BASE = set(_nltk_stopwords.words("english")) | set(_PUNCT)

_REGEX_DIGIT = re.compile(r"^\d+$")
_REGEX_SINGLE_CHAR = re.compile(r"^.{1}$")


def tokenise_base(text):
    """Course-pipeline tokenisation with only base (NLTK + punctuation) stopwords."""
    if not isinstance(text, str):
        return []
    tokens = _tokenizer.tokenize(text.lower())
    out = []
    for tok in tokens:
        tok = tok.strip()
        if tok in _STOPWORDS_BASE:
            continue
        if _REGEX_DIGIT.match(tok):
            continue
        if _REGEX_SINGLE_CHAR.match(tok):
            continue
        out.append(_lemmatizer.lemmatize(tok))
    return out


print(f"Base stopword set size (NLTK English + punctuation): {len(_STOPWORDS_BASE)}")


In [ ]:
# Apply the base tokeniser and count tokens.
rd_clean_df["tokens_base"] = rd_clean_df["clean_text"].apply(tokenise_base)

base_counter = Counter()
for toks in rd_clean_df["tokens_base"]:
    base_counter.update(toks)

print(f"Unique tokens after base stopwords only: {len(base_counter):,}\n")
print("Top-30 tokens with ONLY base stopwords removed:\n")
n_docs = len(rd_clean_df)
for i, (tok, n) in enumerate(base_counter.most_common(30), 1):
    pct = n / n_docs * 100
    print(f"  {i:3d}. {tok:25s} {n:>7,}   (in ~{pct:.1f}% of comments)")


## 7 · Domain-specific stopwords

Inspect the top-30 from cell above and edit `domain_stopwords_final` below. The
generic-English filler core is **identical to notebook 01** (so vocabulary analyses
in nb 03 are directly comparable across platforms); the platform-noise block is
Reddit-specific (`edit`, `op`, `lol`, `lmao`, …) rather than YouTube-specific.

**Important:** topic-essential terms (`iran`, `oil`, `hormuz`, `jet`, `fuel`, `gas`,
`strait`, `usa`, `uk`, `eu`, …) are deliberately NOT included even though they
appear at the top of the list — they *are* the topic.


In [ ]:
# Decided from inspection of the top-30 print in section 6.
#
# The generic-English block is identical to notebook 01's so the
# vocabulary comparisons in nb 03 are platform-agnostic. The platform-noise
# block is Reddit-specific (forum / Markdown UI artefacts) rather than the
# YouTube-UI noise that nb 01 strips.
domain_stopwords_final = [
    # Generic Reddit / conversational fillers
    "u", "us",
    "like", "would", "could", "one", "get", "got",
    "think", "going", "go", "make", "made",
    "people", "time", "thing", "things",
    "know", "want", "much", "many", "also",
    "even", "really", "still", "way", "ways",
    "say", "see", "look", "come", "take", "give",
    "good", "well", "back", "lot", "ever", "let",

    # Reddit / platform-specific noise
    "reddit", "subreddit", "thread", "post", "posts",
    "comment", "comments", "op", "edit", "deleted", "removed",
]

_STOPWORDS_FINAL = _STOPWORDS_BASE | set(domain_stopwords_final)


def tokenise_final(text):
    """Course-pipeline tokenisation with base + domain-specific stopwords."""
    if not isinstance(text, str):
        return []
    tokens = _tokenizer.tokenize(text.lower())
    out = []
    for tok in tokens:
        tok = tok.strip()
        if tok in _STOPWORDS_FINAL:
            continue
        if _REGEX_DIGIT.match(tok):
            continue
        if _REGEX_SINGLE_CHAR.match(tok):
            continue
        out.append(_lemmatizer.lemmatize(tok))
    return out


# Apply final tokenisation.
rd_clean_df["tokens"] = rd_clean_df["clean_text"].apply(tokenise_final)
rd_clean_df["token_count"] = rd_clean_df["tokens"].apply(len)
rd_clean_df["lemma_text"] = rd_clean_df["tokens"].apply(lambda toks: " ".join(toks))
rd_clean_df["char_count"] = rd_clean_df["clean_text"].str.len()

# Drop comments that have zero tokens after final stopwording.
before = len(rd_clean_df)
rd_clean_df = rd_clean_df[rd_clean_df["token_count"] > 0].copy()
attrition.append({"step": "non_empty_after_final_tokens", "comments": int(len(rd_clean_df))})

# Free the intermediate column.
rd_clean_df = rd_clean_df.drop(columns=["tokens_base"], errors="ignore")

# Verify with another top-30 print — should now read as a Hormuz/oil/jet-fuel
# Reddit top-30, not a generic-English-plus-Markdown top-30.
final_counter = Counter()
for toks in rd_clean_df["tokens"]:
    final_counter.update(toks)

print(f"Removed {before - len(rd_clean_df):,} comments that have 0 tokens after final stopwording")
print(f"Final comment count: {len(rd_clean_df):,}\n")
print(f"Unique tokens after final stopwording: {len(final_counter):,}\n")
print("Top-30 tokens AFTER domain stopwords:\n")
n_docs = len(rd_clean_df)
for i, (tok, n) in enumerate(final_counter.most_common(30), 1):
    pct = n / n_docs * 100
    print(f"  {i:3d}. {tok:25s} {n:>7,}   (in ~{pct:.1f}% of comments)")


## 8 · Worked example — one comment, every stage

Pick one comment with at least some interesting features (URL / acronym / repeated
punctuation) and trace it through every preprocessing stage.


In [ ]:
# Pick a comment with interesting features.
candidates = rd_clean_df[
    rd_clean_df["raw_text"].str.contains(r"[!?]{2,}|U\.S\.|http|\bUS\b", regex=True, na=False)
]
example = candidates.iloc[0] if not candidates.empty else rd_clean_df.iloc[0]

print("Pipeline trace on a single Reddit comment\n" + "=" * 60)
print(f"\n  raw_text          : {example['raw_text'][:240]!r}")
print(f"\n  clean_text        : {example['clean_text'][:240]!r}")
print(f"\n  vader_text        : {example['vader_text'][:240]!r}")
print(f"\n  tokens            : {example['tokens'][:25]}")
print(f"\n  lemma_text        : {example['lemma_text'][:240]!r}")
print(f"\n  author_hash       : {example['author_hash']}   (HMAC-SHA256, 16 hex)")
print(f"  parent_author_hash: {example['parent_author_hash']}")
print(f"  subreddit         : {example['subreddit']}")
print(f"  depth             : {example['depth']}")
print(f"  score             : {example['score']}")
print(f"  language          : {example['language']}")
print(f"  is_submitter      : {example['is_submitter']}")
print(f"  is_removed        : {example['is_removed']}")
print(f"  token_count       : {example['token_count']}")
print(f"  char_count        : {example['char_count']}")


## 9 · Visualisations

Five course-relevant diagnostics — same five as notebook 01, with two
Reddit-specific adaptations:

1. **Top-30 token bar chart.**
2. **Character-length and token-count histograms.**
3. **Comments per subreddit and reply-depth distribution** (two subplots) — replaces
   nb 01's comments-per-video. The reply-depth plot is the single best visualisation
   of *why* Reddit is the project's primary network substrate.
4. **Word cloud.**
5. **Comments over time.**


In [ ]:
# Plot 1 — Top-30 token bar chart
top_words = final_counter.most_common(30)
words, counts = zip(*top_words)
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(range(len(words)), counts, color=sns.color_palette("crest", n_colors=len(words)))
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words)
ax.invert_yaxis()
ax.set_xlabel("Frequency")
ax.set_title("Top-30 lemma tokens in the cleaned Reddit corpus")
fig.tight_layout()
fig.savefig(PLOTS / "rd_preproc_top30_tokens.png", dpi=120)
plt.show()


In [ ]:
# Plot 2 — Character length + token count histograms
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(rd_clean_df["char_count"].clip(upper=1500), bins=60,
             color="steelblue", edgecolor="white")
axes[0].set_xlabel("Characters in clean_text (clipped at 1500)")
axes[0].set_ylabel("Comments")
axes[0].set_title("Comment-length distribution")
axes[1].hist(rd_clean_df["token_count"].clip(upper=150), bins=60,
             color="darkorange", edgecolor="white")
axes[1].set_xlabel("Tokens (clipped at 150)")
axes[1].set_ylabel("Comments")
axes[1].set_title("Token-count distribution")
fig.tight_layout()
fig.savefig(PLOTS / "rd_preproc_length_distributions.png", dpi=120)
plt.show()

print(f"clean_text chars   — mean {rd_clean_df['char_count'].mean():.0f}, "
      f"median {rd_clean_df['char_count'].median():.0f}, "
      f"P95 {rd_clean_df['char_count'].quantile(0.95):.0f}")
print(f"token_count        — mean {rd_clean_df['token_count'].mean():.1f}, "
      f"median {rd_clean_df['token_count'].median():.0f}, "
      f"P95 {rd_clean_df['token_count'].quantile(0.95):.0f}")


In [ ]:
# Plot 3 — Comments per subreddit + reply-depth distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 3a. Comments per subreddit
per_sub = rd_clean_df["subreddit"].value_counts()
axes[0].barh(range(len(per_sub)), per_sub.values,
             color=sns.color_palette("rocket", n_colors=len(per_sub)))
axes[0].set_yticks(range(len(per_sub)))
axes[0].set_yticklabels(per_sub.index)
axes[0].invert_yaxis()
axes[0].set_xlabel("Comments (post-cleaning, English only)")
axes[0].set_title("Comments per subreddit")

# 3b. Reply-depth distribution (Reddit-specific)
depths = rd_clean_df["depth"].clip(upper=8).value_counts().sort_index()
axes[1].bar(depths.index.astype(int), depths.values,
            color=sns.color_palette("crest", n_colors=len(depths)))
axes[1].set_xlabel("Reply depth (top-coded at 8)")
axes[1].set_ylabel("Comments")
axes[1].set_title("Reply-depth distribution — why Reddit is our primary network substrate")

fig.tight_layout()
fig.savefig(PLOTS / "rd_preproc_subreddit_and_depth.png", dpi=120)
plt.show()


In [ ]:
# Plot 4 — Word cloud over the cleaned lemma stream
all_tokens = " ".join(rd_clean_df["lemma_text"].tolist())
wc = WordCloud(
    width=1100, height=520,
    background_color="white",
    max_words=200,
    colormap="viridis",
    collocations=False,
).generate(all_tokens)

fig, ax = plt.subplots(figsize=(13, 6))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word cloud — cleaned Reddit lemma tokens", fontsize=14)
fig.tight_layout()
fig.savefig(PLOTS / "rd_preproc_wordcloud.png", dpi=120)
plt.show()


In [ ]:
# Plot 5 — Comments over time
daily = (rd_clean_df.dropna(subset=["created_at"])
         .assign(day=lambda d: d["created_at"].dt.tz_convert("UTC").dt.date)
         .groupby("day").size())

fig, ax = plt.subplots(figsize=(13, 4))
daily.plot(ax=ax, marker="o", color="steelblue")
ax.set_xlabel("Date")
ax.set_ylabel("Comments per day (English, cleaned)")
ax.set_title("Reddit comment volume across the project window")
fig.tight_layout()
fig.savefig(PLOTS / "rd_preproc_volume_over_time.png", dpi=120)
plt.show()


## 10 · Write outputs and summary JSON

Artefacts written to `data/processed/01_preprocessed/`:

- `reddit_comments.{csv,parquet}` — the final comment table (CSV stores `tokens` as
  a space-joined string; parquet preserves the list).
- `reddit_submissions.csv` — one row per submission.
- `reddit_preprocessing_summary.json` — full attrition table, collection metadata
  copied through from the raw JSON, and the domain stopword list used.


In [ ]:
# Final dataframe — Reddit schema. Two extra columns vs the YouTube table:
#   parent_author_hash (essential for the reply graph in nb 04)
#   and the Reddit metadata cluster (depth, score, controversiality, is_submitter,
#   distinguished, is_removed, comment_name).
output_columns = [
    "platform", "submission_id", "subreddit",
    "comment_id", "comment_name", "parent_id",
    "author_hash", "parent_author_hash",
    "raw_text", "clean_text", "vader_text",
    "language",
    "depth", "is_submitter", "distinguished", "score", "controversiality", "is_removed",
    "created_utc", "created_at", "date",
    "tokens", "lemma_text", "token_count", "char_count",
]
rd_final_df = rd_clean_df[output_columns].copy()

# CSV — tokens as space-joined string (matches notebook 01).
csv_df = rd_final_df.copy()
csv_df["tokens"] = csv_df["tokens"].apply(lambda v: " ".join(v) if isinstance(v, list) else "")
csv_df.to_csv(PREPROCESSED / "reddit_comments.csv", index=False)

# Parquet — preserves the list.
try:
    rd_final_df.to_parquet(PREPROCESSED / "reddit_comments.parquet", index=False)
    parquet_written = True
except Exception as exc:
    print(f"[warn] parquet write skipped: {exc}")
    parquet_written = False

# Submissions table.
rd_subs_df.to_csv(PREPROCESSED / "reddit_submissions.csv", index=False)

# Summary JSON with provenance + attrition.
summary = {
    "generated_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "raw_files": [str(p.relative_to(PROJECT_ROOT)) for p in raw_paths],
    "collection_metadata": collection_meta,
    "project_window": {
        "start": PROJECT_START_DATE.isoformat(),
        "end": PROJECT_END_DATE.isoformat(),
    },
    "domain_stopwords": list(domain_stopwords_final),
    "reddit_bot_noise_filter": reddit_bot_noise_summary,
    "attrition": attrition,
    "final_counts": {
        "submissions": int(len(rd_subs_df)),
        "comments": int(len(rd_final_df)),
        "unique_authors": int(rd_final_df["author_hash"].nunique()),
        "english_comments": int((rd_final_df["language"] == "en").sum()),
        "top_level_comments": int((rd_final_df["depth"] == 0).sum()),
        "nested_comments": int((rd_final_df["depth"] > 0).sum()),
        "max_depth": int(rd_final_df["depth"].max()) if not rd_final_df.empty else 0,
        "removed_comments_with_text": int(rd_final_df["is_removed"].fillna(False).sum()),
    },
    "subreddits": rd_final_df["subreddit"].value_counts().to_dict(),
    "outputs": {
        "comments_csv": str((PREPROCESSED / "reddit_comments.csv").relative_to(PROJECT_ROOT)),
        "comments_parquet": str((PREPROCESSED / "reddit_comments.parquet").relative_to(PROJECT_ROOT)) if parquet_written else None,
        "submissions_csv": str((PREPROCESSED / "reddit_submissions.csv").relative_to(PROJECT_ROOT)),
    },
}
(PREPROCESSED / "reddit_preprocessing_summary.json").write_text(json.dumps(summary, indent=2))

# Render the attrition table.
print("Attrition table\n" + "=" * 50)
prev = None
for step in attrition:
    delta = "" if prev is None else f"  (Δ {step['comments'] - prev:+,})"
    print(f"  {step['step']:32s} {step['comments']:>7,}{delta}")
    prev = step["comments"]

print()
print(f"Wrote:")
print(f"  {PREPROCESSED / 'reddit_comments.csv'}")
if parquet_written:
    print(f"  {PREPROCESSED / 'reddit_comments.parquet'}")
print(f"  {PREPROCESSED / 'reddit_submissions.csv'}")
print(f"  {PREPROCESSED / 'reddit_preprocessing_summary.json'}")


## 11 · Limitations specific to this preprocessing step

Same shape as the limitations footer in notebook 01, adapted for Reddit:

- **Language detection on short text.** `detect_language_improved` assumes English for
  any Latin-script comment under 50 characters. Reddit comments skew longer than YouTube
  comments, so this matters less here, but the bias is still acknowledged.

- **Noun-default WordNet lemmatisation.** `WordNetLemmatizer.lemmatize(t)` defaults to
  `pos="n"`, so noun plurals collapse correctly (`barrels` → `barrel`) but verb forms
  do not. We accept the asymmetry for consistency with Assignment 1 and notebook 01.

- **Dedup decision (B1b).** We removed exact + near-duplicate comments but did **not**
  apply a per-author cap. Reddit karma punishes copy-paste so duplicates are rarer than
  on YouTube, but a few prolific accounts can still over-contribute to the corpus.

- **Project-window cutoff.** Comments outside `2026-03-01 .. 2026-05-15` UTC are dropped
  before NLP.

- **`is_removed` comments.** Reddit's moderator-removed comments often have body
  `"[removed]"` or `"[deleted]"` and are filtered by the empty-after-cleaning step.
  We preserve the `is_removed` flag so downstream analyses can report attrition or
  inspect the small fraction of removed comments whose body still contains text.


- **Moderation and bot-template comments.** AutoModerator / VisualMod comments and
  strong subreddit governance templates are removed before text cleaning because they
  are platform artefacts rather than user discourse. The filter is intentionally
  conservative and does not remove ordinary comments just because they contain words
  like `removed` or `bot`.

- **Unresolved `morechildren` placeholders.** The Reddit collector caps the number of
  `morechildren` expansion rounds per thread. The raw JSON's
  `unresolved_more_count` records how many sub-trees were not expanded; this is reported
  in the limitations summary doc.

- **Domain stopwords are research-judgement decisions.** The list in section 7 was
  chosen by inspecting the base-tokenised top-30. We deliberately retain topic-essential
  terms (`iran`, `oil`, `hormuz`, `jet`, `fuel`, …) even though they are very frequent,
  because they *are* the topic. The generic-English core of the list is identical to
  notebook 01's, so the vocabulary analysis in nb 03 stays platform-comparable.
